# 🚀 Colab G4 Studio · NVIDIA RTX PRO 6000 Blackwell (96GB VRAM) 网页端直连工作台

本笔记本用于在不触发第三方客户端「Android 手机安全码」的情况下，直接在已登录 Google Colab Pro 的浏览器中拉起 **NVIDIA RTX PRO 6000 Blackwell (`G4`, `96GB VRAM`)** 算力实例，并生成 Cloudflare 公网加密隧道地址供本地控制台（`http://127.0.0.1:18080`）一键接管。

### ⚡ 使用步骤：
1. 确认顶部菜单 **`Runtime (代码执行程序)`** ➔ **`Change runtime type (更改运行时类型)`** 已选择 **`G4 GPU`**。
2. 如果您要共享主账号已下载好的 134GB Google Drive 模型库，请在本地控制台点击「**📋 一键复制网页端启动代码**」粘贴到下方单元格运行；或者直接点击下方单元格左侧的 **▶** 运行。
3. 跑完后复制底部打印出的 `https://xxxx.trycloudflare.com` 地址，粘贴回本地控制台 `http://127.0.0.1:18080` 点击「**⚡ 立即连接接管**」即可！

In [ ]:
#@title 1️⃣ 检查当前分配的 GPU (确认是否为 NVIDIA RTX PRO 6000 Blackwell 96GB)
!nvidia-smi

In [ ]:
#@title 2️⃣ 一键启动 Colab G4 Studio 推理与视频超分双隧道服务
# 💡 提示：若您从本地控制台 (http://127.0.0.1:18080) 复制了带有 134GB 云盘凭证的启动代码，可直接覆盖粘贴到此单元格运行！
import os, subprocess, sys

MODEL_REPO = "MiniMaxAI/MiniMax-H3" #@param ["MiniMaxAI/MiniMax-H3", "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF", "unsloth/Qwen3.8-27B-GGUF"]
MODEL_FOLDER = "MiniMax-H3" #@param ["MiniMax-H3", "Huihui-Qwen3.8-27B-abliterated-GGUF", "Qwen3.8-27B-GGUF"]

subprocess.run("curl -fsSL https://downloads.rclone.org/rclone-current-linux-amd64.zip -o /tmp/rc.zip && unzip -qo /tmp/rc.zip -d /tmp && cp /tmp/rclone-*-linux-amd64/rclone /usr/local/bin/rclone && chmod +x /usr/local/bin/rclone", shell=True, check=False)
if not os.path.isdir("/content/colab-g4-studio"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/wwintj/colab-g4-studio.git", "/content/colab-g4-studio"], check=True)
else:
    subprocess.run(["git", "-C", "/content/colab-g4-studio", "pull", "--ff-only"], check=False)

env = os.environ.copy()
pattern = "FL2VA/*.safetensors" if "minimax" in MODEL_REPO.lower() else "*.gguf"
env.update({"HF_REPO": MODEL_REPO, "MODEL_FOLDER": MODEL_FOLDER, "FILE_PATTERN": pattern})
p = subprocess.Popen([sys.executable, "-u", "/content/colab-g4-studio/deploy_colab_model.py"], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end="", flush=True)
p.wait()

In [ ]:
#@title 3️⃣ 用完后一键断开并释放 G4 实例（立即停止扣除 CU 算力单元）
from google.colab import runtime
print("🛑 正在释放当前 Colab G4 实例，停止计费...")
runtime.unassign()